# 02 — Greeks Analysis

Price the enriched position set and calculate instrument-level risk measures: NPV, Delta, Gamma, Vega, Theta, and Rho.

This notebook uses the same pricing functions as the CLI and test suite.


## 1. Notebook setup


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.6f}".format)

DATA_DIR = PROJECT_ROOT / "data" / "processed"
DATA_DIR.mkdir(parents=True, exist_ok=True)


## 2. Build market context

`MarketContext` centralizes valuation date, curves, volatility assumptions, and FX foreign rates.


In [ ]:
from src.config.settings import RiskSettings
from src.pricing.instruments import MarketContext

settings = RiskSettings()
ctx = MarketContext(
    valuation_date=pd.Timestamp(settings.valuation_date),
    risk_free_rate=settings.risk_free_rate,
    dividend_yield=settings.dividend_yield,
    vols=settings.vols(),
    fx_foreign_rates={"EURUSD=X": 0.015, "GBPUSD=X": 0.018},
)
ctx


## 3. Load enriched positions

If the CSV does not exist yet, regenerate it using the production data modules.


In [ ]:
from src.data.generate_positions import generate_synthetic_positions
from src.data.generate_structure import generate_structure
from src.data.market_data import enrich_positions_with_market, load_market_data

positions_path = DATA_DIR / "positions_enriched.csv"

if positions_path.exists():
    enriched = pd.read_csv(positions_path, parse_dates=["Maturity"])
else:
    positions = generate_synthetic_positions(settings.valuation_date)
    structure = generate_structure()
    market = load_market_data(settings.tickers, settings.valuation_date, seed=settings.random_seed)
    enriched = enrich_positions_with_market(positions, structure, market, settings.valuation_date)

enriched.head()


## 4. Calculate instrument risk

Stocks use direct mark-to-market. FX forwards use interest-rate parity. European options use the pure-Python Black-Scholes implementation, with a QuantLib integration boundary available for future extension.


In [ ]:
from src.pricing.greeks import calculate_instrument_risk, GREEK_COLUMNS

instrument_risk = calculate_instrument_risk(enriched, ctx)

cols = ["PositionID", "InstrumentType", "Ticker", "Quantity", "TradingDesk", "Unit", *GREEK_COLUMNS]
instrument_risk[cols].round(6)


## 5. Desk-level Greek aggregation

Greeks are additive across positions for this simplified framework. VaR is handled separately via full revaluation, not Greek summation.


In [ ]:
from src.aggregation.hierarchy import aggregate_greeks

desk_greeks = aggregate_greeks(instrument_risk, ["TradingDesk"])
unit_greeks = aggregate_greeks(instrument_risk, ["Unit"])

display(desk_greeks.round(6))
display(unit_greeks.round(6))


## 6. Sanity checks

These checks make the notebook suitable for a production project review: no missing risk values, stock Delta equals quantity, and option Vega is finite.


In [ ]:
assert instrument_risk[GREEK_COLUMNS].notna().all().all(), "Missing Greek values"

stock_rows = instrument_risk["InstrumentType"].eq("Stock")
assert np.allclose(
    instrument_risk.loc[stock_rows, "Delta"],
    instrument_risk.loc[stock_rows, "Quantity"],
), "Stock Delta should equal quantity"

option_rows = instrument_risk["InstrumentType"].eq("European Option")
assert np.isfinite(instrument_risk.loc[option_rows, "Vega"]).all(), "Option Vega must be finite"

print("Risk checks passed")


## 7. Persist instrument-level risk


In [ ]:
output_path = DATA_DIR / "instrument_risk.csv"
instrument_risk.to_csv(output_path, index=False)
print(f"Wrote {output_path.relative_to(PROJECT_ROOT)}")


## 8. Risk visualization

Plot the largest absolute Delta contributors to quickly identify directional exposure concentration.


In [ ]:
import matplotlib.pyplot as plt

plot_df = instrument_risk.assign(AbsDelta=instrument_risk["Delta"].abs()).sort_values("AbsDelta", ascending=False)
ax = plot_df.plot.bar(x="PositionID", y="AbsDelta", figsize=(11, 5), legend=False, title="Largest Delta Contributors")
ax.set_xlabel("Position")
ax.set_ylabel("Absolute Delta")
plt.tight_layout()
